In [1]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [2]:
import pickle
import os

def load_wesad_subject(file_path):
    with open(file_path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')
    return data


In [3]:
from scipy.signal import resample  # This is the correct import

def resample_signal(signal, target_len):
    return resample(signal, target_len, axis=0)

In [4]:
def extract_signals_labels_with_threshold(data, 
                                          chest_sensors=['ACC', 'ECG', 'Resp'], 
                                          wrist_sensors=['ACC', 'EDA', 'TEMP', 'BVP'],
                                          window_size=128,
                                          overlap=0.5,
                                          normalize=True):
    from collections import Counter
    import numpy as np
    from scipy.signal import resample

    chest = data['signal']['chest']
    wrist = data['signal']['wrist']
    raw_labels = data['label']

    # --- Step 1: Find minimum wrist length ---
    wrist_len = min([
        len(wrist[s]) for s in wrist_sensors if s in wrist
    ])

    # --- Step 2: Resample labels from 700Hz -> 64Hz ---
    labels_resampled = resample(raw_labels, wrist_len)
    labels_resampled = np.round(labels_resampled).astype(int)  # Round to nearest integer
    labels = labels_resampled

    # --- Step 3: Cut wrist signals to wrist length ---
    wrist_data = np.concatenate([
        wrist[s][:wrist_len] for s in wrist_sensors if s in wrist
    ], axis=1)

    # --- Step 4: Resample chest signals to wrist length ---
    chest_data = np.concatenate([
        resample(chest[s], wrist_len).reshape(wrist_len, -1)
        for s in chest_sensors if s in chest
    ], axis=1)

    # --- Step 5: Remove undefined labels (0) ---
    valid_mask = labels > 0

    wrist_data = wrist_data[valid_mask]
    chest_data = chest_data[valid_mask]
    labels = labels[valid_mask]

    # --- Step 6: Relabel: Stress (2) -> 1, Others -> 0 ---
    labels = np.where(labels == 2, 1, 0)

    print("→ Converted binary label distribution:", np.unique(labels, return_counts=True))

    # --- Step 7: Normalize features ---
    if normalize and len(chest_data) > 0 and len(wrist_data) > 0:
        chest_data = (chest_data - chest_data.mean(axis=0)) / (chest_data.std(axis=0) + 1e-6)
        wrist_data = (wrist_data - wrist_data.mean(axis=0)) / (wrist_data.std(axis=0) + 1e-6)

    all_data = np.concatenate([chest_data, wrist_data], axis=1)

    # --- Step 8: Sliding window ---
    step_size = max(1, int(window_size * (1 - overlap)))

    X, y = [], []
    for start in range(0, len(all_data) - window_size + 1, step_size):
        end = start + window_size
        window = all_data[start:end]
        label_window = labels[start:end]

        window_label = 1 if np.any(label_window == 1) else 0

        X.append(window)
        y.append(window_label)

    print("✅ Window-level label distribution:", Counter(y))

    return X, y


In [17]:
import numpy as np
import os
from collections import Counter

def load_multiscale_wesad_data_for_federated_learning(base_path='WESAD/', 
                                                      window_sizes=[128, 256, 512],
                                                      overlap=0.25,
                                                      subject_ids=None,
                                                      chest_sensors=['ACC', 'ECG', 'Resp'], 
                                                      wrist_sensors=['ACC', 'EDA', 'TEMP', 'BVP'],
                                                      normalize=True):
    if subject_ids is None:
        subject_ids = [i for i in range(2, 18) if i != 12]

    client_data = []

    for subject_id in subject_ids:
        print(f"\n📦 Processing subject S{subject_id}")
        file_path = os.path.join(base_path, f'S{subject_id}', f'S{subject_id}.pkl')
        data = load_wesad_subject(file_path)

        X_multi = []
        y_multi = []

        for window_size in window_sizes:
            print(f"  🔍 Window size: {window_size}")
            X, y = extract_signals_labels_with_threshold(
                data,
                chest_sensors=chest_sensors,
                wrist_sensors=wrist_sensors,
                window_size=window_size,
                overlap=overlap,
                normalize=normalize
            )
            if len(X) == 0:
                print(f"⚠️ Skipping scale {window_size} for S{subject_id} (no valid windows)")
                break
            X_multi.append(np.array(X))
            y_multi.append(np.array(y))

        if len(X_multi) != len(window_sizes):
            print(f"⚠️ Skipping S{subject_id} (not all scales available)")
            continue

        # Find minimal number of windows
        min_len = min([len(y) for y in y_multi])

        # Align all scales to min_len
        X_multi = [X[:min_len] for X in X_multi]
        y_multi = [y[:min_len] for y in y_multi]

        # Merge labels across scales
        # Any window with at least one stress label across scales is considered stress
        y_stacked = np.stack(y_multi, axis=1)  # shape: (min_len, num_scales)
        y_final = (np.sum(y_stacked, axis=1) > 0).astype(int)

        if len(np.unique(y_final)) < 2:
            print(f"⚠️ Subject S{subject_id} after merge has only one class, skipping...")
            continue

        print(f"✅ S{subject_id}: Final shape per scale {[X.shape for X in X_multi]}, Labels: {np.unique(y_final, return_counts=True)}")
        client_data.append((X_multi, y_final))

    print(f"\n✅ Loaded {len(client_data)} clients.")
    return client_data


In [18]:
clients = load_multiscale_wesad_data_for_federated_learning(window_sizes=[32, 64, 128], overlap=0.25)



📦 Processing subject S2
  🔍 Window size: 32
→ Converted binary label distribution: (array([0, 1]), array([9616, 2462]))
✅ Window-level label distribution: Counter({0: 397, 1: 105})
  🔍 Window size: 64
→ Converted binary label distribution: (array([0, 1]), array([9616, 2462]))
✅ Window-level label distribution: Counter({0: 198, 1: 53})
  🔍 Window size: 128
→ Converted binary label distribution: (array([0, 1]), array([9616, 2462]))
✅ Window-level label distribution: Counter({0: 97, 1: 28})
✅ S2: Final shape per scale [(125, 32, 11), (125, 64, 11), (125, 128, 11)], Labels: (array([0, 1]), array([68, 57]))

📦 Processing subject S3
  🔍 Window size: 32
→ Converted binary label distribution: (array([0, 1]), array([10012,  2560]))
✅ Window-level label distribution: Counter({0: 415, 1: 108})
  🔍 Window size: 64
→ Converted binary label distribution: (array([0, 1]), array([10012,  2560]))
✅ Window-level label distribution: Counter({0: 206, 1: 55})
  🔍 Window size: 128
→ Converted binary label d

In [19]:
import torch
import torch.nn as nn

class ModalityTransformer(nn.Module):
    def __init__(self, input_dim, embed_dim=64, nhead=4, num_layers=2):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=nhead, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x):  # x: [B, T, input_dim]
        x = self.input_proj(x)
        x = self.encoder(x)  # [B, T, embed_dim]
        return x.mean(dim=1)  # [B, embed_dim]

In [20]:
class AttentionFusion(nn.Module):
    def __init__(self, embed_dim=64, num_modalities=3):
        super().__init__()
        self.attn_weights = nn.Parameter(torch.randn(num_modalities, embed_dim))

    def forward(self, embeddings):  # [B, M, D]
        B, M, D = embeddings.shape
        attn_scores = torch.einsum('bmd,md->bm', embeddings, self.attn_weights)
        attn_scores = torch.softmax(attn_scores, dim=1)
        weighted = (attn_scores.unsqueeze(-1) * embeddings).sum(dim=1)  # [B, D]
        return weighted


In [21]:
class MultiScaleFusionTransformer(nn.Module):
    def __init__(self, input_dims, embed_dim=64, num_classes=3):
        super().__init__()
        self.encoders = nn.ModuleList([
            ModalityTransformer(d, embed_dim) for d in input_dims
        ])
        self.fusion = AttentionFusion(embed_dim=embed_dim, num_modalities=len(input_dims))
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )

    def forward(self, inputs):  # list of tensors: len = num_scales
        embeds = [encoder(x) for encoder, x in zip(self.encoders, inputs)]  # [B, D]
        stacked = torch.stack(embeds, dim=1)  # [B, S, D]
        fused = self.fusion(stacked)  # [B, D]
        return self.classifier(fused)


In [22]:
def train_local(model, data, epochs=3, lr=1e-3, device='cpu'):
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    X_multi, y = data
    X_multi = [torch.tensor(x, dtype=torch.float32).to(device) for x in X_multi]
    y = torch.tensor(y, dtype=torch.long).to(device)

    dataset = torch.utils.data.TensorDataset(*X_multi, y)
    loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)

    for _ in range(epochs):
        for batch in loader:
            *inputs, labels = [b.to(device) for b in batch]
            optimizer.zero_grad()
            output = model(inputs)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()


In [23]:
import random
import copy

def federated_round(global_model, client_data, frac=0.4, device='cpu'):
    selected_clients = random.sample(client_data, int(len(client_data) * frac))

    local_weights = []
    for data in selected_clients:
        local_model = copy.deepcopy(global_model).to(device)
        train_local(local_model, data, device=device)
        local_weights.append({k: v.cpu() for k, v in local_model.state_dict().items()})

    # FedAvg
    avg_weights = {}
    for key in local_weights[0].keys():
        avg_weights[key] = sum([w[key] for w in local_weights]) / len(local_weights)

    global_model.load_state_dict(avg_weights)
    return global_model



In [24]:
from sklearn.metrics import classification_report, accuracy_score
import torch

@torch.no_grad()
def validate_model(model, client_data, device='cpu'):
    model.eval()

    y_true_all = []
    y_pred_all = []

    for X_multi, y in client_data:
        X_multi = [torch.tensor(x, dtype=torch.float32).to(device) for x in X_multi]
        y = torch.tensor(y, dtype=torch.long).to(device)

        batch_size = 64
        num_samples = y.shape[0]
        
        for i in range(0, num_samples, batch_size):
            inputs = [x[i:i+batch_size] for x in X_multi]
            labels = y[i:i+batch_size]
            outputs = model(inputs)
            preds = outputs.argmax(dim=1)

            y_true_all.extend(labels.cpu().numpy())
            y_pred_all.extend(preds.cpu().numpy())

    acc = accuracy_score(y_true_all, y_pred_all)
    print(f"✅ Validation Accuracy: {acc:.4f}")
    print(classification_report(y_true_all, y_pred_all, digits=4))
    return acc



In [25]:
#clients = load_multiscale_wesad_data_for_federated_learning(window_sizes=[128, 256, 512])
input_dims = [X.shape[2] for X in clients[0][0]]

model = MultiScaleFusionTransformer(input_dims=input_dims).to('cuda')

In [28]:
for rnd in range(1000):
    print(f"\n🌐 Federated Round {rnd + 1}")
    model = federated_round(model, clients, frac=0.5, device='cuda')
    print(f"🔍 Evaluation after round {rnd + 1}")
    validate_model(model, clients, device='cuda')  # You can also pass a separate test split



🌐 Federated Round 1
🔍 Evaluation after round 1
✅ Validation Accuracy: 0.8302
              precision    recall  f1-score   support

           0     0.8664    0.8790    0.8727      1298
           1     0.7562    0.7345    0.7452       663

    accuracy                         0.8302      1961
   macro avg     0.8113    0.8068    0.8089      1961
weighted avg     0.8291    0.8302    0.8296      1961


🌐 Federated Round 2
🔍 Evaluation after round 2
✅ Validation Accuracy: 0.8144
              precision    recall  f1-score   support

           0     0.8360    0.8952    0.8646      1298
           1     0.7618    0.6561    0.7050       663

    accuracy                         0.8144      1961
   macro avg     0.7989    0.7757    0.7848      1961
weighted avg     0.8109    0.8144    0.8106      1961


🌐 Federated Round 3
🔍 Evaluation after round 3
✅ Validation Accuracy: 0.8302
              precision    recall  f1-score   support

           0     0.8863    0.8529    0.8693      1298
   

In [29]:
import torch

torch.save(model.state_dict(), "federated_tsbert.pth")
print("✅ Model saved as federated_tsbert.pth")

✅ Model saved as federated_tsbert.pth
